# 01 从 Prompt 到 Action

## 大模型为什么看起来像在执行任务

大模型最容易让人产生误解的地方，不是它会回答问题，而是它看起来像是在“做事”。当一个模型能够整理需求、选择工具、输出结构化参数，甚至在多轮上下文里持续推进任务时，直觉上很容易把它理解成一个已经具备执行力的智能体。

这个判断并不准确，但也不完全错。

不准确的部分在于，模型本身并没有像传统程序那样拥有显式控制流、任务状态机和函数执行器。它仍然是一个基于上下文进行 next-token prediction 的生成系统。看起来像“执行”的行为，本质上仍然发生在生成过程中。

不完全错的部分在于，一旦上下文被设计成一个任务环境，生成过程就不再只是补全文字，而会逐渐表现出类似决策、规划和动作选择的外观。也正是在这里，Prompt、System Prompt、Tool Schema、Runtime、Agent 和 MCP 开始进入同一个问题域。

这一章不讨论 MCP 协议本身，也不急着讨论 Agent Runtime 的完整结构。这里先回答一个更底层的问题：**为什么一个文本生成模型，会在工程系统里表现得像一个任务执行者。**

## 先给结论

如果把这件事压缩成一句话，可以这样说：

> 大模型并不是先学会了“执行任务”，再被接到工具系统里；相反，它先学会了在复杂上下文里延续最像任务执行过程的语言结构，工程系统再把这些结构解释成真正的动作。

这句话里有三个关键点。

- 第一，模型最底层仍然是生成器，不是传统意义上的控制器。
- 第二，所谓指令跟随，本质上是上下文条件变化之后的输出分布变化。
- 第三，所谓动作执行，并不是模型自己完成的，而是外部运行时把模型输出解释成系统动作之后才成立。

这三个点，决定了后面所有 Agent 系统的设计边界。

## 1. 模型不是规则引擎，但会表现出规则感

很多入门材料会把“指令跟随”说成模型听懂了命令。这个说法方便传播，但对工程理解帮助不大。

更准确的说法是：模型在大量训练语料、指令微调数据和偏好对齐数据中，学会了把某一类输入上下文映射成某一类输出风格。输入里一旦出现任务描述、角色约束、格式要求、成功标准，模型就会倾向于续写出符合这些条件的回应。

这不是因为模型内部突然长出了一套 if/else 规则树，而是因为这些上下文特征会系统性改变下一个 token 的概率分布。

从这个角度看，所谓“听话”并不是模型服从命令，而是模型在统计意义上学会了：

- 什么时候应该解释
- 什么时候应该列步骤
- 什么时候应该输出 JSON
- 什么时候应该承认信息不足
- 什么时候应该先做中间推理再给最终答案

这也解释了一个常见现象：模型行为往往像规则驱动，但又永远不会像编译器那样稳定。因为它给出的不是确定性执行结果，而是概率最高的语言延续。

## 2. 指令跟随，本质上是条件续写

大模型之所以能从“自由续写”变成“任务响应”，关键不是参数规模本身，而是训练目标和训练数据发生了变化。

在预训练阶段，模型主要学习的是一般性的语言结构、知识关联和上下文延续能力。到了指令微调阶段，输入不再只是原始文本片段，而开始包含：

- 用户请求
- 角色设定
- 输出格式要求
- 示例问答
- 带偏好信号的高质量响应

这会带来一个重要结果：模型逐渐把“看到一个任务描述”理解成“接下来应该输出一段符合任务结构的内容”。于是，原本只是文本预测的问题，开始表现出任务响应的轮廓。

这里最值得强调的一点是，**指令跟随不等于理解了命令式程序语义，指令跟随只是模型学会了什么样的上下文后面通常应该接什么样的文本。**

这个 distinction 很重要。因为后面当模型进入 tool calling、Agent loop、MCP resource read 的场景时，工程上最危险的误判，就是把“很像理解”直接等同于“可以放心执行”。

## 3. 为什么它会表现得像在行动

如果模型只是续写文本，为什么会出现“先分析问题，再决定调用工具，然后补全答案”这种强烈的行动感？原因在于，任务环境本身已经把语言空间重塑成了动作空间。

一旦上下文里包含以下要素：

- 明确的目标
- 清晰的角色约束
- 可以调用的工具描述
- 可读取的外部信息
- 对结果格式的显式要求

模型就不再是在一个开放式写作环境里生成文本，而是在一个受限任务环境里生成“下一步最合理的系统输出”。

这时它输出的内容虽然仍然是 token，但这些 token 的语义地位已经发生了变化：

- 一段自然语言解释，可能是决策理由
- 一段 JSON，可能是工具参数
- 一个函数名，可能是动作选择
- 一段结构化状态，可能是下一轮执行的输入

模型不是突然获得了手和脚，而是外部系统把它的文本输出映射成了动作接口。所谓“行动能力”，正是在这层映射中被制造出来的。

## 4. 格式约束，本质上是在把语言变成接口

从工程角度看，Prompt 的真正价值不只是“说服模型”，而是给模型的输出空间加边界。格式约束越明确，输出就越容易被程序系统消费。

这也是为什么 system prompt、few-shot、JSON schema、tool definition 这些看似分散的技术，实际上在做同一件事：**把开放语言压缩成可解释、可验证、可执行的结构。**

比如，当模型被要求：

- 先判断是否需要外部信息
- 如果需要，则必须返回工具名和参数
- 如果不需要，则直接给答案

它面对的就不再是“随便怎么答都行”的问题，而是“当前上下文下，哪一种输出结构最符合约束”。

从这里开始，Prompt 不再只是文案技巧，而开始变成运行时接口设计的一部分。

一个最小示意可以直接写清楚：

- `system`：你是任务执行型助手；当问题需要外部信息时，不要猜测，而要返回结构化工具调用意图。
- `user`：请确认今天 Indianapolis 的天气，并判断是否适合晚上跑步。
- `tool schema`：存在一个 `get_weather` 工具，接收 `location` 和可选 `date` 这类参数。
- `model next output`：不是直接写天气结论，而是先给出类似“调用 `get_weather(location=Indianapolis, date=today)`”的动作意图。

这里最重要的不是 JSON 长什么样，而是这一步已经体现出：模型在当前上下文里优先生成了“系统可解释的下一步动作”，而不是继续自然语言补全。

上面这个例子有一个很关键的观察点：`tool_call_intent` 仍然只是文本结构化之后的结果，它本身并没有真的去查天气。

真正发生的事情其实分成两层：

- 模型层：判断“这个问题不能直接猜，需要外部信息”，并输出一个结构化调用意图
- 运行时层：读取这个结构化意图，真的去执行 `get_weather`，把结果再送回模型

也就是说，**模型负责生成动作意图，系统负责兑现动作后果。**

这就是从 Prompt 走向 Action 的第一道边界。很多演示把这两层混在一起，于是看上去像是模型天然会“调用工具”。实际上，工具调用永远是运行时对模型输出的解释结果。

## 5. System Prompt 为什么是第一层控制面

在这个阶段，system prompt 的作用还不是让模型“更聪明”，而是让模型在多个可能输出里更稳定地偏向任务型输出。

如果没有 system prompt，用户的一句话可能被理解成开放聊天，也可能被理解成信息检索请求，还可能被理解成创意写作任务。模型会在多个风格分布之间摇摆。

一旦加入 system prompt，事情就变了。system prompt 实际上在做三件事：

- 定义当前会话的身份和工作模式
- 规定遇到不确定信息时应该怎么处理
- 决定模型更倾向于直接回答、先解释、还是先调用外部能力

从这个意义上说，system prompt 不是装饰信息，而是整个运行时的第一层控制面。后面无论是 tool calling 还是 MCP resource 读取，本质上都依赖这一层来建立行为边界。

## 6. 这还不是 Agent，但已经是 Agent 的起点

到这里为止，模型已经表现出几种容易被误认为“智能体”的特征：

- 能根据目标调整输出风格
- 能在不知道答案时倾向于请求外部能力
- 能输出看起来像动作的结构化内容

但这仍然不能直接称为 Agent。

原因很简单：真正的 Agent 至少还需要以下几个额外层次：

- 持续的任务状态
- 多步循环而不是单步回应
- 对外部能力的显式调度
- 对结果的回填和再决策
- 终止条件与错误恢复

换句话说，本章解释的是 Agent 为什么可能出现，而不是 Agent 已经成立。Prompt 把模型推进到任务空间，Tool Schema 把语言输出压缩成动作接口，真正把这一切组织成闭环的，是后面的 Agent Runtime。

把这条最小链路直接写成流程，会更清楚：

1. `system prompt` 定义任务边界。
2. `user message` 提供当前目标。
3. 模型输出自然语言内容，或者结构化动作意图。
4. Runtime 解释这个意图，并决定是否访问外部能力。
5. 外部结果被回填进上下文。
6. 模型基于更新后的上下文继续生成下一步输出。

这六步本身还不是完整 Agent，但已经是从 Prompt 走向 Action 的最小闭环骨架。

## 7. 为什么这一章必须先讲

如果一开始就直接进入 MCP 或 Agent 框架，很容易把问题理解反了：仿佛是先有工具协议，模型才获得能力；或者先有 Agent 编排，模型才表现出任务性。

实际顺序恰好相反。

先发生的是模型在复杂上下文中表现出结构化响应能力，然后工程系统才利用这种能力，把输出解释成指令、接口和动作。MCP 解决的是这些外部能力如何被标准化暴露，Agent 解决的是这些能力如何被任务化调度，但二者都建立在同一个前提上：**模型已经能够在上下文里稳定地产生可被系统消费的结构化输出。**

所以这一章的意义，不是重复“大模型会预测下一个 token”这个基础事实，而是把这件事实和后面的工程结构真正连起来。只有这样，后面讨论 tool calling、Agent loop 和 MCP 时，才不会滑向“API 拼装教程”的语气。

## 8. 本章结论

这一章要建立的判断可以压缩成四句话：

- 大模型首先是生成系统，不是执行系统。
- 指令跟随本质上是上下文条件变化后的输出分布变化。
- 当输出格式被约束时，语言就开始具备接口属性。
- 当运行时把这些结构化输出解释成系统动作时，模型才表现出“像在执行任务”的外观。

后面的章节会继续把这个链条往下推进：先讲 system prompt 和 message roles 如何构成控制面，再讲 tool calling 如何把动作意图转成交互闭环，最后再进入 Agent Runtime 与 MCP 的关系。